In [1]:
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import pandas as pd

In [2]:
INPUT_FILE = "list.csv"
OUTPUT_FILE = "list.csv"

In [3]:
session = requests.Session()

In [4]:
TIMEOUT = 15

def gen_header(PORTAL, MAC):
    return {
        "User-Agent": "Mozilla/5.0 (QtEmbedded; U; Linux; C)",
        "X-User-Agent": "Model: MAG250; Link: WiFi",
        "Referer": f"{PORTAL}/c/",
        "Cookie": f"mac={MAC}; stb_lang=en; timezone=UTC",
    }

def handshake(PORTAL, MAC):
    url = f"{PORTAL}/portal.php"
    params = {"type": "stb", "action": "handshake", "token": "", "JsHttpRequest": "1-xml"}
    r = session.get(url, headers=gen_header(PORTAL, MAC), params=params, timeout=TIMEOUT, verify=False,)
    r.raise_for_status()
    data = r.json()
    token = data["js"]["token"]
    return token

def get_account_info(PORTAL, MAC, token):
    url = f"{PORTAL}/portal.php"
    headers = gen_header(PORTAL, MAC)
    headers["Authorization"] = f"Bearer {token}"
    params = {"type": "account_info", "action": "get_main_info", "JsHttpRequest": "1-xml"}
    r = session.get(url, headers=headers, params=params, timeout=TIMEOUT, verify=False)
    r.raise_for_status()
    return r.json()

def get_expiry(PORTAL, MAC):
    token = handshake(PORTAL, MAC)
    account = get_account_info(PORTAL, MAC, token)
    js = account.get("js", {})
    return (js.get("end_date") or js.get("phone") or js.get("tariff_plan") or js.get("account_expiration"))

In [5]:
df = pd.read_csv(INPUT_FILE, dtype=str)

In [6]:
results = []
for _, row in df.iterrows():
    result = row.copy()
    try:
        result['Expiry'] = get_expiry(row['PORTAL'], row['MAC'])
    except:
        result['Expiry'] = "Invalid"
    results.append(result)

In [7]:
pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)